# Análisis de series temporales
En el presente cuadernillo se hallan los modelos de análisis de series temporales, hecho en `R`, para el caso de análisis de este TFG. El análisis completo de los datos solo se realizará en este cuadernillo, mientras que en el cuadernillo <a href="Análisis en Python.ipynb" target="_blank"><code>Análisis en Python.ipynb</code></a> solo se replica el modelo y se realiza la comparación sintáctica para usarse como base para el uso del paquete `pyrcore` en la última parte del TFG.

**Caso de estudio**: Análisis temporal de las series de producción de gas de esquisto en varios sitios de EEUU para el periodo enero de 2000 a agosto de 2022 (2000.01-2022.08).

In [ ]:
# Comprobar si se han instalado los siguientes paquetes:
# DBI
if (!require("DBI")){
    # En caso contrario, instalar y cargar
    install.packages("DBI")
    library(DBI)
}

# RSQLite
if (!require("RSQLite")){
    # En caso contrario, instalar y cargar
    install.packages("RSQLite")
    library(RSQLite)
}

# ggplot2
if (!require("ggplot2")){
    # En caso contrario, instalar y cargar
    install.packages("ggplot2")
    library(ggplot2)
}

# forecast
if (!require("forecast")){
    # En caso contrario, instalar y cargar
    install.packages("forecast")
    library(forecast)
}

## Análisis de series temporales univariantes
Análisis temporal de las series de producción de gas de esquisto en varios sitios de EEUU para el periodo enero de 2000 a agosto de 2022 (2000.01-2022.08).

### Importación de datos
Extraer los datos necesarios del fichero *DataBase* `Base de datos TFG.db` en forma de serie temporal (`ts()`).

In [ ]:
# Conectar a la base de datos
conn <- dbConnect(RSQLite::SQLite(), "..\\datasets\\Base de datos TFG.db")
# Importar datos desde la base de datos
gas <- dbGetQuery(conn, "SELECT * FROM Esquisto")
# Retirar la última columna, que no es una serie temporal
gas <- gas[-length(colnames(gas))]
# Desconectar de la base de datos
dbDisconnect(conn)
# Estadísticos básicos
summary(gas)
# Previsualizar datos
head(gas)

In [ ]:
# Establecer las fechas como los nombres de las filas
rownames(gas) <- gas$Date # gas[, 1]
# Quitar las fechas del data.frame
gas <- gas[-1]
# Previsualizar de nuevo
head(gas)

Dado que se replica este trabajo de la asignatura *Data Modelling: Econometría*, se ha de elegir una de la series. En este caso, elegiré la primera serie temporal, del grupo *Barnett (TX)*.

In [ ]:
# Extraer Barnett (TX) como serie temporal
gas <- ts( # Función "ts()" para creación de series temporales
    gas[1], # Columna de "Barnett (TX)"
    start = 2000, # Comienza en enero del año 2000
    frequency = 12 # Datos mensuales
)
# Visualizar serie temporal
gas

### Análisis clásico
Análisis clásico de series temporales univariantes.

#### Identificación del esquema
En primer lugar, se realiza un gráfico media-desviación típica para identificar si la serie sigue un esquema aditivo o multiplicativo.

In [ ]:
# Hallar promedios y desviaciones típicas anuales
medias <- c(aggregate(gas, nfrequency = 1, FUN = mean))
desviaciones <- c(aggregate(gas, nfrequency = 1, FUN = sd))

# Unificar en un data.frame
media_desviacion <- data.frame(
    Media = medias, # Columna "Media"
    Desviación_típica = desviaciones, # Columna "Desviación_típica"
    row.names = 2000:2021 # Nombres de las filas (solo filas completas)
)
media_desviacion

In [ ]:
# Ajustar tamaño de la ventana del dispositivo de gráficos
options(repr.plot.width = 10, repr.plot.height = 7.5)

In [ ]:
# Representar la media frente a la desviación típica anuales
ggplot(
    media_desviacion, # data.frame con promedios y desviaciones anuales
    aes(x = Media, y= Desviación_típica) # Ejes
) + geom_line() +
labs(
    title = "Media-Desviación",
    y = "Desviación Típica"
) +
theme(plot.title = element_text(hjust = 0.5))
ggsave("..\\Gráficos\\Media-Desviación.png", width = 10, height = 7.5)

Dado que la desviación típica varía a medida que lo hace la media, se concluye que la serie sigue un esquema **multiplicativo**.

#### Tendencia
Se representan gráficamente los datos para comprobar si existe una tendencia y, en tal caso, su forma.

In [ ]:
# Cambiar fondo a blanco
par(bg = "white")
# Generar gráfico
plot(
  gas, # Datos
  main = "Producción de gas de esquisto de Barnett (TX)", # Título
  xlab = "Año", # Etiqueta eje X
  ylab = "Producción (en miles de millones de pies cúbicos)" # Etiqueta eje Y
)
# Cuadrícula
grid(col = "darkgray", lty = "dashed")

# Almacenar gráfico
grafico <- recordPlot()

# Preparar ruta y tamaño para guardar el gráfico
png("..\\Gráficos\\Serie temporal.png", width = 1000, height = 750)

# Reproducir gráfico para guardarlo
replayPlot(grafico)

# Cerrar archivo .png
dev.off()

# Liberar espacio en memoria borrando la variable gráfico
rm(grafico)

Parece observarse una tendencia cuadrática.

#### Descomposición
Se procede a la descomposición en la tendencia y componente estacional.

In [ ]:
# Descomponer serie temporal
descomposicion <- decompose(gas, type = "multiplicative")
# Obtener tendencia
tendencia <- descomposicion$trend
# Cálculo de Índices de Variación Estacional (IVEs)
ives <- tapply(descomposicion$seasonal, cycle(descomposicion$seasonal), mean)
# Mostrar tendencia e IVEs
tendencia
print(ives)

Los **Í**ndices de **V**ariación **E**stacional (**IVE**s) son extremadamente cercanos a 1, por lo que no hay componente estacional significativa.

Aquí concluye el análisis clásico y se da paso a la metodología *Box-Jenkins*.

### Metodología *Box-Jenkins*

#### Estudiar la estacionariedad de la serie

##### Media

En el análisis clásico pareció observarse una tendencia cuadrática. Se procede a la comprobación de la tendencia.

In [ ]:
# Usando el vector "medias"
medias

In [ ]:
# Representar, mediante un diagrama de líneas, los promedios anuales
par(bg = "white")
plot(
  2000:2021, medias,
  type = "l",
  main = "Evolución de la media por año",
  xlab = "Año",
  ylab = "Media"
)
grid(col = "darkgray", lty = "dashed")
grafico <- recordPlot()
png("..\\Gráficos\\Media-Tiempo.png", width = 1000, height = 750)
replayPlot(grafico)
dev.off()
rm(grafico)

En efecto, se observa una tendencia cuadrática.

La serie **no es estacionaria en media** y, como la tendencia es cuadrática, requerirá aplicar **dos diferencias lineales**.

##### Varianza

Representar gráficamente la evolución de la varianza respecto al tiempo para comprobar si es <ins>heterocedástica</ins> (no estacionaria en varianza) u <ins>homocedástica</ins> (estacionaria en varianza).

In [ ]:
# Usando el vector desviaciones
desviaciones ^ 2

In [ ]:
# Representar, mediante un diagrama de líneas, las varianzas anuales
par(bg = "white")
plot(
  2000:2021, desviaciones ^ 2,
  type = "l",
  main = "Evolución anual de la varianza",
  xlab = "Año",
  ylab = "Varianza"
)
grid(col = "darkgray", lty = "dashed")
grafico <- recordPlot()
png("..\\Gráficos\\Varianza-Tiempo.png", width = 1000, height = 750)
replayPlot(grafico)
dev.off()
rm(grafico)

La varianza no es constante a lo largo del tiempo. Sin embargo, dado que no existe un patrón en las variaciones de la varianza, se considera que estas fluctuaciones se deben a elementos accidentales.

**<ins>NO</ins>** se requiere **transformación logarítmica**. El cálculo de $\lambda$ para estabilización de la varianza no ha sido aprendido durante el curso. Afortunadamente, existen funciones para integrar su uso en la modelización como, por ejemplo, la transformación de Box-Cox.

##### Componente estacional
Dado que no hay componente estacional significativa, no se requiren transformaciones estacionales.

#### Transformación de la serie

##### Separación

In [ ]:
# Conjunto de entrenamiento
train <- window(gas, end = c(2022, 3))

# Conjunto de prueba
test <- window(gas, start = c(2022, 4))

# Visualizar
train
test

##### Diferencia lineal
Fórmula matemática:
$\Delta y_t = y_t - y_{t-1}$

$\Delta^d y_t = y_t - y_{t-1} - y_{t-2} - ... - y_{t-d}$

In [ ]:
# DOS diferencias lineales
train <- diff(train, differences = 2)
train

#### Identificación del modelo

##### Correlogramas

In [ ]:
# Función de Autocorrelación
par(bg = "white")
acf(
    train,
    main = "Función de Autocorrelación de la serie Barnett (TX)",
    xlab = "Retardo"
)
grafico <- recordPlot()
png("..\\Gráficos\\ACF.png", width = 1000, height = 750)
replayPlot(grafico)
dev.off()
rm(grafico)

In [ ]:
# Función de Autocorrelación Parcial
par(bg = "white")
pacf(
    train,
    main = "Función de Autocorrelación Parcial de la serie Barnett (TX)",
    xlab = "Retardo", ylab = "PACF"
)
grafico <- recordPlot()
png("..\\Gráficos\\PACF.png", width = 1000, height = 750)
replayPlot(grafico)
dev.off()
rm(grafico)

**MA(q=2)**: La ACF empieza en negativo, pasa ligeramente a positivo y se corta abruptamente a partir de ahí. La PACF presenta una forma de onda sinusoidal.

Por tanto, el modelo es un **MA(2)** (`q = 2`).

#### Estimación de los coeficientes de los modelos

Se ajusta un modelo ARIMA a la **serie transformada**.

In [ ]:
# Modelo MA(2)
modelo <- arima(train, order = c(0, 0, 2))
modelo

#### Contraste de validez del modelo
Evalúa el modelo utilizando métricas como AIC, RMSE, etc.

In [ ]:
print(paste("AIC:", AIC(modelo)))
print(paste("BIC:", BIC(modelo)))

In [ ]:
pred <- forecast(modelo, h = length(test))
accuracy(pred)

#### Análisis detallado de los errores

In [ ]:
Box.test(
    residuals(modelo),
    lag = log(length(train)) * 10,
    type = "Ljung-Box",
    fitdf = 4
)

p-valor < $\alpha$ => no es ruido blanco. Probablemente se deba a la ausencia de la transformación de Box-Cox. A continuación se hace una comprobación con `auto.arima()`.

In [ ]:
# Comprobación
Box.test(
    residuals(
        auto.arima(
            window(gas, end = c(2022, 3)), # Conjunto de train sin transformar
            lambda = "auto" # Cálculo automático de lambda
        )
    ),
    lag = log(length(train)) * 10,
    type = "Ljung-Box",
    fitdf = 4
)

p-valor > $\alpha$ => no es ruido blanco.

Véase que `auto.arima()` establece que la serie temporal responde a un modelo `ARIMA(0, 2, 2)`.

In [ ]:
auto.arima(window(gas, end = c(2022, 3)), lambda = "auto")

#### Predicción

In [ ]:
pred <- forecast(modelo)
par(bg = "white")
plot(pred)
grafico <- recordPlot()
png("..\\Gráficos\\Predicción.png", width = 1000, height = 750)
replayPlot(grafico)
dev.off()
rm(grafico)
accuracy(pred, test)

El modelo falla enormemente en predecir datos. Ahora apliquemos la misma celda al modelo hecho con `auto.arima()`.

In [ ]:
train <- window(gas, end = c(2022, 3))
modelo <- auto.arima(train, lambda = "auto")
pred <- forecast(modelo)
par(bg = "white")
plot(pred)
grafico <- recordPlot()
png("..\\Gráficos\\Predicción.png", width = 1000, height = 750)
replayPlot(grafico)
dev.off()
rm(grafico)
accuracy(pred, test)

El modelo es significativamente mejor para predecir datos. $\lambda$ es lo que arruina la **Metodología Box-Jenkins** <ins>manual</ins>, dado que su cálculo nunca se aprendió en ninguna asignatura, sin perjuicio de que un servidor sea capaz de dilucidar este asunto.

El análisis ha terminado y el modelo **ajusta apropiadamente**.